In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# 1. Project setup
# -----------------------------
np.random.seed(42)

output_dir = Path("data/clean")
output_dir.mkdir(parents=True, exist_ok=True)

instruments = ["EURUSD", "GBPUSD", "AUDUSD"]
start_prices = {
    "EURUSD": 1.0850,
    "GBPUSD": 1.2700,
    "AUDUSD": 0.6750
}

dates = pd.bdate_range(start="2025-01-02", periods=45)

# Daily return assumptions (simple normal noise)
daily_vol = {
    "EURUSD": 0.0035,
    "GBPUSD": 0.0040,
    "AUDUSD": 0.0045
}

# Possible trade sizes in base currency units
trade_sizes = [25000, 50000, 75000, 100000, 150000]

# -----------------------------
# 2. Generate prices.csv
# -----------------------------
price_rows = []

for instrument in instruments:
    prior_close = start_prices[instrument]
    
    for i, date in enumerate(dates):
        if i == 0:
            close_price = prior_close * (1 + np.random.normal(0, daily_vol[instrument]))
        else:
            close_price = price_rows[-1]["close_price"] * (1 + np.random.normal(0, daily_vol[instrument]))
        
        price_rows.append({
            "date": date.strftime("%Y-%m-%d"),
            "instrument": instrument,
            "close_price": round(close_price, 5),
            "prior_close_price": round(prior_close, 5)
        })
        
        prior_close = close_price

prices = pd.DataFrame(price_rows)

# -----------------------------
# 3. Generate trades.csv
# -----------------------------
trade_rows = []
trade_id_counter = 1

for date in dates:
    date_str = date.strftime("%Y-%m-%d")
    
    for instrument in instruments:
        # Random number of trades for this instrument/date
        n_trades = np.random.choice([0, 1, 2, 3], p=[0.20, 0.40, 0.25, 0.15])
        
        close_price = prices.loc[
            (prices["date"] == date_str) & (prices["instrument"] == instrument),
            "close_price"
        ].iloc[0]
        
        for _ in range(n_trades):
            side = np.random.choice(["BUY", "SELL"])
            quantity = np.random.choice(trade_sizes)
            
            # Trade price near close, slightly noisy
            trade_price = close_price * (1 + np.random.normal(0, 0.0008))
            
            trade_rows.append({
                "trade_id": f"T{trade_id_counter:05d}",
                "date": date_str,
                "instrument": instrument,
                "side": side,
                "quantity": quantity,
                "trade_price": round(trade_price, 5)
            })
            
            trade_id_counter += 1

trades = pd.DataFrame(trade_rows)

# -----------------------------
# 4. Generate fees.csv
# -----------------------------
fee_rows = []

for _, row in trades.iterrows():
    # Slightly size-sensitive fee logic
    base_fee = 2.00
    variable_fee = row["quantity"] / 50000 * np.random.uniform(0.75, 1.50)
    fee_amount = round(base_fee + variable_fee, 2)
    
    fee_rows.append({
        "trade_id": row["trade_id"],
        "fee_amount": fee_amount
    })

fees = pd.DataFrame(fee_rows)

# -----------------------------
# 5. Generate positions.csv
# -----------------------------
position_rows = []

# Track prior end position for each instrument
prior_positions = {instrument: 0 for instrument in instruments}

for date in dates:
    date_str = date.strftime("%Y-%m-%d")
    
    for instrument in instruments:
        start_position = prior_positions[instrument]
        
        day_trades = trades[
            (trades["date"] == date_str) & (trades["instrument"] == instrument)
        ].copy()
        
        if not day_trades.empty:
            day_trades["signed_qty"] = np.where(
                day_trades["side"] == "BUY",
                day_trades["quantity"],
                -day_trades["quantity"]
            )
            net_trade_qty = day_trades["signed_qty"].sum()
        else:
            net_trade_qty = 0
        
        end_position = start_position + net_trade_qty
        
        position_rows.append({
            "date": date_str,
            "instrument": instrument,
            "start_position": start_position,
            "end_position": end_position
        })
        
        prior_positions[instrument] = end_position

positions = pd.DataFrame(position_rows)

# -----------------------------
# 6. Validation checks
# -----------------------------
# A. trade_id uniqueness
assert trades["trade_id"].is_unique, "Duplicate trade_id values found."

# B. every trade has exactly one fee
assert len(trades) == len(fees), "Mismatch between trades and fees row counts."
assert set(trades["trade_id"]) == set(fees["trade_id"]), "Some trades are missing fees or vice versa."

# C. one row per date/instrument in prices and positions
assert not prices.duplicated(subset=["date", "instrument"]).any(), "Duplicate date/instrument in prices."
assert not positions.duplicated(subset=["date", "instrument"]).any(), "Duplicate date/instrument in positions."

# D. position continuity check
positions_sorted = positions.sort_values(["instrument", "date"]).copy()

for instrument in instruments:
    inst_positions = positions_sorted[positions_sorted["instrument"] == instrument].reset_index(drop=True)
    for i in range(1, len(inst_positions)):
        prev_end = inst_positions.loc[i - 1, "end_position"]
        curr_start = inst_positions.loc[i, "start_position"]
        assert prev_end == curr_start, f"Position continuity error for {instrument} on row {i}"

# -----------------------------
# 7. Save CSV files
# -----------------------------
prices.to_csv(output_dir / "prices.csv", index=False)
trades.to_csv(output_dir / "trades.csv", index=False)
fees.to_csv(output_dir / "fees.csv", index=False)
positions.to_csv(output_dir / "positions.csv", index=False)

print("Clean datasets created successfully.")
print(f"prices rows: {len(prices)}")
print(f"trades rows: {len(trades)}")
print(f"fees rows: {len(fees)}")
print(f"positions rows: {len(positions)}")

Clean datasets created successfully.
prices rows: 135
trades rows: 200
fees rows: 200
positions rows: 135
